In [2]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go

import fastplotlib as fpl

# Init

In [3]:
time = 100
dt = 0.005
steps = int(time / dt)

time = 100
dt = 0.005

# Transient for washing out start of lorentz
# Transient for washing out fresh springs
steps = int(time / dt)
transient_steps_lorentz = int(steps * 0.05)
transient_steps_springs = int(steps * 0.05)
total_steps = steps + transient_steps_lorentz + transient_steps_springs

test_size = 0.2

t = np.linspace(0, total_steps * dt, total_steps)

In [4]:
sigma, rho, beta = 10, 28, 8.0 / 3.0

initial_state = [1.0, 1.0, 1.0]

def lorenz_system(t, state, sigma=sigma, rho=rho, beta=beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

sol = solve_ivp(
    lorenz_system,
    (0, total_steps * dt),
    initial_state,
    t_eval=t,
)

lorentz_dataset = sol.y.T[transient_steps_lorentz:]

mean_l = np.mean(lorentz_dataset, axis=0)
std_l = np.std(lorentz_dataset, axis=0)
lorenz_scaled = (lorentz_dataset - mean_l) / std_l

In [5]:
def lorentz_plot(data):
    fig = go.Figure(
        data=go.Scatter3d(
            x=data[:, 0],
            y=data[:, 1],
            z=data[:, 2],
            mode="lines",
            line=dict(color="blue", width=2),
        )
    )
    return fig

In [6]:
fig = lorentz_plot(lorentz_dataset)
fig.show()

In [7]:
def create_stiffness_matrix(node_positions, connections):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for conn in connections:
        node_conn = conn[:2].astype(int)
        node_pos = node_positions[node_conn]
        k_val = conn[2]

        diff_vec = np.diff(node_pos, axis=0).flatten()
        unit_dir = diff_vec / np.linalg.norm(diff_vec)

        sub_block = np.outer(unit_dir, unit_dir)
        k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])

        global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")

        for local_row, global_row in enumerate(global_indices):
            for local_col, global_col in enumerate(global_indices):
                K[global_row, global_col] += k_local[local_row, local_col]

    return K

In [8]:
@njit
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [9]:
def ridge_regression(X, Y, random_split = True):
    scaler = StandardScaler()

    if random_split:
        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y, test_size=test_size, random_state=42
        )
    else:
        X_train, X_test = X[:int(len(X) * (1 - test_size))], X[int(len(X) * (1 - test_size)):]
        Y_train, Y_test = Y[:int(len(Y) * (1 - test_size))], Y[int(len(Y) * (1 - test_size)):]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RidgeCV()
    model.fit(X_train_scaled, Y_train)
    Y_pred = model.predict(X_test_scaled)

    return model, (Y_test, Y_pred)

In [68]:
def spring_animation(
    disp, nodes_pos, connections_list, size=15, external=False, is_3d=False, frames_moved=5, max_frames=2000
):
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(cameras="3d", controller_types="orbit", canvas="glfw" if external else "jupyter")
    )

    dots = fig[0, 0].add_scatter(
        data=nodes_pos_3d.astype(np.float32), sizes=size, colors="magenta"
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack(
                [nodes_pos_3d[int(row[0])], nodes_pos_3d[int(row[1])]]
            ).astype(np.float32),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]
        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

    fig.add_animations(update_springs)
    return fig

In [66]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

# Test Chain with U from -1 to 1

In [25]:
N = 10
nodes_pos = np.arange(0, N).reshape(-1, 1)
node_ids = np.arange(N)

In [26]:
u_val = 1
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [27]:
rng = np.random.default_rng(42)

tau_steps = int(.7 / dt)

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N-1)

transient_steps = transient_steps_lorentz + transient_steps_springs
U = np.zeros((steps + transient_steps, matrix_size))
U[:, 0] = u

In [28]:
connections_list = np.column_stack((node_ids[:-1], node_ids[1:], k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [29]:
disp, v = run_simulation(steps + transient_steps, dt, matrix_size, M_INV, DAMP, K, U)
disp = disp[transient_steps:]
v = v[transient_steps:]
Y = u[transient_steps:]

In [30]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(X[tau_steps:], Y[:-tau_steps])

In [31]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9119 0.2968


In [32]:
y_pred_discrete = np.where(Y_pred > 0.0, u_val, -u_val)
r_2 = r2_score(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, y_pred_discrete) * 100
print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9119 99.40%


In [41]:
fig = weight_plot(model.coef_)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list)
fig.show()

RFBOutputContext()

# Chain with Lorentz

In [60]:
N = 30
nodes_pos = np.arange(0, N).reshape(-1, 1)
node_ids = np.arange(N)

In [61]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N - 1)

U = np.zeros((steps + transient_steps_springs, matrix_size))
U[:, 0] = lorenz_scaled[:, 0]
U[:, int(N/2)] = lorenz_scaled[:, 1]
U[:, -1] = lorenz_scaled[:, 2]

In [62]:
connections_list = np.column_stack((node_ids[:-1], node_ids[1:], k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [63]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
u = lorenz_scaled[transient_steps_springs:]

In [91]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(X[:-tau_steps], u[tau_steps:], False)

In [92]:
r_2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.8482 0.1512


In [93]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()

In [94]:
fig = spring_animation(disp, nodes_pos, connections_list)
fig.show()

RFBOutputContext()

In [ ]:
fig = lorentz_plot(Y_pred)
fig.show()

## Is it better to use random or sequential test split

In [97]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(X[:-tau_steps], u[tau_steps:], True)

In [98]:
r_2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.8638 0.1401


Seems to get a better score but what about against the future or a sequential split

In [101]:
scaler = StandardScaler()

X_data = X[:-tau_steps]
u_data = u[tau_steps:]
X_train, X_test = (
    X_data[: int(len(X_data) * (1 - test_size))],
    X_data[int(len(X_data) * (1 - test_size)) :],
)
Y_train, Y_test = (
    u_data[: int(len(u_data) * (1 - test_size))],
    u_data[int(len(u_data) * (1 - test_size)) :],
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
Y_pred = model.predict(X_test_scaled)

In [102]:
r_2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6703 0.3256


In [ ]:
fig = lorentz_plot(Y_pred)
fig.show()

So use sequential split

# Test Ring with U from -1 to 1

In [42]:
N = 30

theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
radius = 10.0
x_pos = radius * np.cos(theta)
y_pos = radius * np.sin(theta)

nodes_pos = np.column_stack((x_pos, y_pos))

In [43]:
u_val = 10
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [44]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 100, size=N)

transient_steps = transient_steps_lorentz + transient_steps_springs
U = np.zeros((steps + transient_steps, matrix_size))
ux = -np.sin(theta[0]) * u
uy = np.cos(theta[0]) * u
U[:, 0] = ux
U[:, 1] = uy

In [45]:
node_ids = np.arange(N)
connections_list = np.column_stack((node_ids, np.roll(node_ids, -1), k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [46]:
disp, v = run_simulation(steps + transient_steps, dt, matrix_size, M_INV, DAMP, K, U)
disp = disp[transient_steps:]
v = v[transient_steps:]
Y = u[transient_steps:]

In [47]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(X[tau_steps:], Y[:-tau_steps])

In [48]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9780 1.4840


In [49]:
y_pred_discrete = np.where(Y_pred > 0.0, u_val, -u_val)

r_2 = r2_score(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9780 99.50%


In [67]:
fig = weight_plot(model.coef_)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, is_3d=True, external=True)
fig.show()

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:400: RuntimeWarning: divide by zero encountered in divide
  screen_full = (ndc_full[:, :2] / ndc_full[:, 3:4]) * half_canvas_size
/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:400: RuntimeWarning: invalid value encountered in divide
  screen_full = (ndc_full[:, :2] / ndc_full[:, 3:4]) * half_canvas_size
/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:412: RuntimeWarning: invalid value encountered in divide
  screen_sel = (ndc_sel[:, :2] / ndc_sel[:, 3:4]) * half_canvas_size


AttributeError: 'ImguiFigure' object has no attribute 'auto_scale'

# Ring with Lorentz

In [164]:
N = 10

theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
radius = 5.0
x_pos = radius * np.cos(theta)
y_pos = radius * np.sin(theta)

nodes_pos = np.column_stack((x_pos, y_pos))

In [196]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N)

U = np.zeros((steps + transient_steps_springs, matrix_size))
U[:, 0] = lorenz_scaled[:, 0]
U[:, int(N / 3)] = lorenz_scaled[:, 1]
U[:, int(2 * N / 3)] = lorenz_scaled[:, 2]

In [197]:
node_ids = np.arange(N)
connections_list = np.column_stack((node_ids, np.roll(node_ids, -1), k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [198]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
u = lorenz_scaled[transient_steps_springs:]

In [199]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred) = ridge_regression(X[:-tau_steps], u[tau_steps:], False)

In [200]:
r_2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.8357 0.1639


In [201]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list)
fig.show()

RFBOutputContext()